In [1]:
from IPython.lib.deepreload import original_reload
!pip install opencv-python

  Obtaining dependency information for opencv-python from https://files.pythonhosted.org/packages/e9/a5/1be1516390333ff9be3a9cb648c9f33df79d5096e5884b5df71a588af463/opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata
  Obtaining dependency information for numpy>=2 from https://files.pythonhosted.org/packages/92/0f/7ceaaeaacb40567071e94dbf2c9480c0ae453d5bb4f52bea3892c39dc83c/numpy-2.4.2-cp312-cp312-win_amd64.whl.metadata
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.2/40.2 MB 5.8 MB/s eta 0:00:07
   - -------------------------------------- 1.1/40.2 MB 13.5 MB/s eta 0:00:03
   - -------------------------------------- 1.9/40.2 MB 15.3 MB/s eta 0:00:03
   -- ------------------------------------- 2.8/40.2 MB 16.3 MB/s eta 0:00:03
   --- ------------------------------------ 3.7/40.2 MB 16.7 MB/s eta 0:00:03
   ---- ----------------------------------- 4.6/40.2 MB 17.1 MB/s eta 0:00:03
   ----- -------------------------


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import cv2
import numpy as np

In [31]:
original_image = cv2.imread("Squirrel.JPG")
#original_image = cv2.imread("Bluebell.JPG")
#original_image = cv2.imread("Cas.jpg")

In [32]:
cv2.imwrite("debug.png", original_image)

True

In [33]:
scales = [original_image]
for i in range(4):
    scales.append(cv2.resize(scales[-1], None, fx=0.5, fy=0.5, interpolation=cv2.INTER_CUBIC))
cv2.imwrite("debug.png", scales[-1])


True

In [34]:
def BlurImages(img):
    #return cv2.bilateralFilter(img, 20, 100, 100)
    blur = cv2.GaussianBlur(img, (21, 21), 4)
    cv2.imwrite("debugBlur.png", blur)
    return blur

def MakeOdd(num):
    if num <= 1:
        print("Kernal Size was too small so set to 3")
        return 3
    return num - num%2 + 1

In [35]:
def FindEdges(sizes, size, pixel_detail_factor, minimum_edge_threshold, diff_from_median_threshold):
    img = sizes[size]
    blurred = BlurImages(img)

    gray = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)

    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)  # Horizontal edges
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)  # Vertical edges

    # Compute gradient magnitude
    gradient_magnitude = cv2.magnitude(sobelx, sobely)

    # Convert to uint8
    gradient_magnitude = cv2.convertScaleAbs(gradient_magnitude)


    edges = gradient_magnitude
    cv2.imwrite("debugEdges.png", edges)



    adujsted_threshold = minimum_edge_threshold * 255




    median_edges = cv2.medianBlur(edges, MakeOdd(128 // (2**size) + 1))

    colour_mask = np.bitwise_and((edges > median_edges * diff_from_median_threshold), (edges > adujsted_threshold)) * 255

    cv2.imwrite("debugMask.png", colour_mask)

    blurred_edges = cv2.GaussianBlur(edges, (15, 15), 0)


    alt_median_edges = cv2.medianBlur(blurred_edges, 128 // (2**size) + 1)

    alt_colour_mask = np.bitwise_and((blurred_edges > alt_median_edges * diff_from_median_threshold), (blurred_edges > adujsted_threshold)) * 255

    cv2.imwrite("altDebugMask.png", alt_colour_mask)

    cv2.imwrite("alt.png", np.append(img, alt_colour_mask[:, :, np.newaxis], axis=2))
    out = np.append(img, colour_mask[:, :, np.newaxis], axis=2)

    return out

In [36]:
cv2.imwrite("debug.png",
            FindEdges(scales, size=1,
                      pixel_detail_factor = 1, # Use this to adjust the relative information per pixel.
                      # debugEdges.JPG
                      #
                      # Edge Thresholding:
                      minimum_edge_threshold = 0.05,
                      diff_from_median_threshold = 1.5
                      # debugMask
                      )
            )

True

In [37]:
def ImageComponentAnalysis(sizes, size):
    img = sizes[size]
    blurred = cv2.medianBlur(img, 7)



    bin_size = 48


    binned_image = blurred - (blurred)%bin_size# + bin_size//2

    cv2.imwrite("ICA.png", binned_image)




In [38]:
ImageComponentAnalysis(scales, 1)

In [39]:
#!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [40]:
#!pip install "transformers[torch]"


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
import torch
import cv2
import numpy as np
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation, SamModel, SamProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [42]:
# Load ADE20K pretrained model
processor = SegformerImageProcessor.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512"
)
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512"
).to(device)

model.eval()

Loading weights: 100%|██████████| 380/380 [00:00<00:00, 1075.03it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [43]:
# Get class names
id2label = model.config.id2label

In [44]:
image_to_segment = scales[1]
image_rgb = cv2.cvtColor(image_to_segment, cv2.COLOR_BGR2RGB)

# Preprocess
inputs = processor(images=image_rgb, return_tensors="pt").to(device)

# Inference
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
upsampled_logits = torch.nn.functional.interpolate(
    logits,
    size=image_rgb.shape[:2],
    mode="bilinear",
    align_corners=False,
)

pred_mask = upsampled_logits.argmax(dim=1)[0].cpu().numpy()

# Create color map
num_classes = model.config.num_labels
colors = np.random.randint(0, 255, (num_classes, 3), dtype=np.uint8)
colored_mask = colors[pred_mask]

colored_mask_bgr = cv2.cvtColor(colored_mask, cv2.COLOR_RGB2BGR)

overlay = cv2.addWeighted(image_to_segment, 0.6, colored_mask_bgr, 0.4, 0)

# ---- ADD CLASS LABELS ----
unique_classes = np.unique(pred_mask)

for class_id in unique_classes:
    if class_id == 0:  # skip background if desired
        continue

    mask = (pred_mask == class_id).astype(np.uint8)

    # Find centroid of the mask
    moments = cv2.moments(mask)
    if moments["m00"] == 0:
        continue

    cx = int(moments["m10"] / moments["m00"])
    cy = int(moments["m01"] / moments["m00"])

    label = id2label[class_id]

    # Draw text with outline for readability
    cv2.putText(overlay, label, (cx, cy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (0, 0, 0), 3, cv2.LINE_AA)
    cv2.putText(overlay, label, (cx, cy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255, 255, 255), 1, cv2.LINE_AA)

cv2.imwrite("Segmentation.png", overlay)


True

In [64]:
model = SamModel.from_pretrained("facebook/sam-vit-large").to(device)
processor = SamProcessor.from_pretrained("facebook/sam-vit-large")



Loading weights: 100%|██████████| 482/482 [00:00<00:00, 997.81it/s, Materializing param=vision_encoder.pos_embed]                                                  


In [52]:
del(img)

NameError: name 'img' is not defined

In [65]:
image_to_segment = scales[1]
image_rgb = cv2.cvtColor(image_to_segment, cv2.COLOR_BGR2RGB)

inputs = processor(images=image_rgb, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Remove extra dimensions carefully
masks = outputs.pred_masks.squeeze(0).squeeze(0)  # remove batch dim
masks = masks.cpu().numpy()            # shape: (num_masks, H, W)

overlay = image_to_segment.copy()
np.random.seed(42)

for i in range(masks.shape[0]):

    mask = masks[i]

    # Ensure mask matches original resolution
    mask_resized = cv2.resize(
        mask,
        (image_to_segment.shape[1], image_to_segment.shape[0]),
        interpolation=cv2.INTER_NEAREST
    )

    binary_mask = mask_resized > 0.5  # boolean mask (H, W)

    color = np.random.randint(0, 255, size=3)

    # Apply mask correctly across 3 channels
    overlay[binary_mask] = (
            0.6 * overlay[binary_mask] +
            0.4 * color
    ).astype(np.uint8)


cv2.imwrite("Segmentation.png", overlay)


True

In [66]:
masks.shape

(3, 256, 256)

In [67]:
outputs.pred_masks.shape

torch.Size([1, 1, 3, 256, 256])